# Helios — Google Colab Training & Deployment Pipeline

Trains the Helios family, exports GGUF, updates Hugging Face + Ollama, and syncs the repo.

**Runtime:** GPU (T4/A100 recommended). Change runtime to GPU before running.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
CKPT_DIR = '/content/drive/MyDrive/Helios'
os.makedirs(CKPT_DIR + '/checkpoints', exist_ok=True)
os.makedirs('/Volumes', exist_ok=True)
if not os.path.exists('/Volumes/One Touch'):
    os.symlink('/content/drive/MyDrive', '/Volumes/One Touch')
os.environ['CKPT_DIR'] = CKPT_DIR + '/checkpoints'
print('Checkpoint dir:', os.environ['CKPT_DIR'], 'exists:', os.path.isdir(os.environ['CKPT_DIR']))
if not os.path.isdir(os.environ['CKPT_DIR']):
    print('>>> Upload your helios_nano/alpha/beta checkpoint folders to Google Drive: MyDrive/Helios/checkpoints/')


In [ ]:
!nvidia-smi || true
!git clone https://github.com/darknecrocities/Helios.git /content/helios 2>/dev/null || (cd /content/helios && git pull)
%cd /content/helios
!git checkout pretrained
!apt-get -qq install -y zstd 2>/dev/null
!pip -q install -e . 2>/dev/null; pip -q install datasets transformers accelerate safetensors huggingface_hub gguf 2>/dev/null
import sys; sys.path.insert(0,'/content/helios')
import torch; print('torch', torch.__version__, 'cuda', torch.cuda.is_available())


In [ ]:
import os
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    os.environ['GITHUB_TOKEN'] = userdata.get('GITHUB_TOKEN')
except Exception:
    pass
from huggingface_hub import login
if os.environ.get('HF_TOKEN'):
    login(token=os.environ['HF_TOKEN'])
    print('HF logged in')
else:
    print('No HF_TOKEN; set it in Colab secrets for push')


In [ ]:
# Optional: pull Helios checkpoints from HF if uploaded
!mkdir -p /content/helios/checkpoints
python - <<'PY'
import os
from huggingface_hub import snapshot_download
repo = os.environ.get('HF_REPO', 'Darknecrocities/Helios')
try:
    snapshot_download(repo_id=repo, local_dir='/content/helios/checkpoints', allow_patterns=['checkpoints/**','tokenizer/**'], token=os.environ.get('HF_TOKEN'))
    print('checkpoints downloaded')
except Exception as e:
    print('checkpoint download skipped:', e)
PY


In [ ]:
import os
tok = os.path.join(os.environ.get('CKPT_DIR','/content/drive/MyDrive/Helios/checkpoints'), 'helios_nano')
if not os.path.isdir(tok):
    print('Tokenizer checkpoint not found at', tok)
    print('Upload helios_nano folder to Drive: MyDrive/Helios/checkpoints/helios_nano')
else:
    !HF_HOME=/content/hf_cache PYTHONPATH=/content/helios python scripts/build_pretrain_shards.py \
      --out-dir /content/datasets/helios_fineweb_v2 \
      --tokenizer "$CKPT_DIR/helios_nano" \
      --source "HuggingFaceFW/fineweb|sample-10BT|train|text|200000|odc-by" \
      --source "HuggingFaceFW/fineweb-edu|sample-10BT|train|text|50000|odc-by" \
      --shard-tokens 20000000 --val-shard-tokens 5000000 --val-every 100 --checkpoint-every 1000
    !PYTHONPATH=/content/helios python scripts/verify_pretrain_shards.py --dataset-dir /content/datasets/helios_fineweb_v2


In [ ]:
import os
ckpt = os.path.join(os.environ.get('CKPT_DIR','/content/drive/MyDrive/Helios/checkpoints'), 'helios_nano')
if os.path.isdir(ckpt):
    !PYTHONPATH=/content/helios python scripts/continue_full_family_lineage.py --tiers nano --seconds 120 --seq-len 256 --batch-size 4 --lr 5e-5
    !ls -la "/Volumes/One Touch/Helios/checkpoints/v050_full_lineage" 2>/dev/null || true
else:
    print('Nano checkpoint missing at', ckpt)
    print('Upload helios_nano folder to Drive: MyDrive/Helios/checkpoints/helios_nano')


In [ ]:
!python -c "print('GGUF export step: implement helios/export/gguf.py export for each tier')"


In [ ]:
import os
token = os.environ.get('GITHUB_TOKEN')
if token:
    !git remote set-url origin https://{token}@github.com/darknecrocities/Helios.git
    !git config user.email 'parejasarronkian@gmail.com'
    !git config user.name 'Arron Kian Parejas'
    !git add -A && git commit -m 'colab: training + export run' || true
    !git push origin pretrained
    print('Repo synced')
else:
    print('No GITHUB_TOKEN; skipping push. Set it in Colab secrets.')


In [ ]:
!apt-get -qq install -y zstd 2>/dev/null; curl -fsSL https://ollama.com/install.sh | sh
!nohup ollama serve >/tmp/ollama.log 2>&1 &
import time; time.sleep(8)
!cd /content/helios && for f in deploy/ollama/Modelfile*; do tag=$(basename $f | sed 's/Modelfile//;s/^\./:/;s/^$/latest/'); echo "Building $tag from $f"; ollama create parejasarronkian/Helios${tag} -f $f || true; done
!ollama list || true
